In [ ]:
import glob, os
from langchain_community.document_loaders import PyMuPDFLoader, TextLoader
docs = []
for path in glob.glob("cvs/*.pdf"):
    docs += PyMuPDFLoader(path).load()
for path in glob.glob("cvs/*.md"):
    docs += TextLoader(path, encoding="utf-8").load()

1 صفحة/ملف


In [134]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
chunks = splitter.split_documents(docs)
print(len(chunks), "chunk")

6 chunk


In [136]:
from langchain_ollama import OllamaEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
embeddings = OllamaEmbeddings(model="nomic-embed-text")
vectorstore = InMemoryVectorStore.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 6, "fetch_k": 15})

In [137]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatOllama(model="gemma3:4b", temperature=1)

SCOPE = """
Name: Ahmed Mohamed Galal (in Arabic: أحمد محمد جلال)
ABOUT ME (English): <اكتب هنا 2-3 جمل من الـ CV بالإنجليزي>     
ABOUT ME (Arabic): <نفس الكلام بالعربي بصياغتك>
What I do: Data science, machine learning, deep learning, NLP, RAG and LLM chatbots.
Tools: Python, SQL, Scikit-learn, PyTorch, TensorFlow, LangChain, Ollama, ...
What I do NOT do: mobile apps, game development, hardware/electronics, graphic design.
"""
OUT_OF_SCOPE = "عذراً، هذا خارج نطاق خدماتي. أقدر أساعدك في الأسئلة عن خبراتي ومشاريعي."
prompt = ChatPromptTemplate([
    ("system",
    "LANGUAGE RULE: You MUST reply ONLY in {language}. "
    "Do not mix in any other language. If {language} is English, write everything in English,"
    "including the name (Ahmed Mohamed Galal) and technical terms.\n"
    "For 'tell me about yourself' questions, use the ABOUT ME text in the requested language "
    "almost verbatim. Do not add anything to it.\n"
     "You are the professional AI assistant for Ahmed Mohamed Galal.\n"
     "The name Ahmed Mohamed Galal must be written in Arabic exactly as: أحمد محمد جلال "
    "(never غالال or any other spelling). In English write it as Ahmed Mohamed Galal.\n"
     "Your job is to answer questions about his CV, skills, experience, and what he can do.\n"
     "Answer ONLY using the provided context (his CV and scope notes). "
     "Keep names and numbers exactly as written.\n"
     "Never invent skills, experience, or prices that are not in the context.\n"
     "Reply in the exact same language as the user (Arabic or English).\n"
     "Always speak in the first person as Ahmed (I, my), never in the third person.\n"
     "How to match the user's request with the context:\n"
     "- Match by MEANING, not only exact words. Treat abbreviations, synonyms and "
     "Arabic/English variants as the same thing "
     "(e.g. ML = Machine Learning, DL = Deep Learning, NLP = Natural Language Processing, "
     "sklearn = Scikit-learn, 'تعلم الآلة' = Machine Learning, 'شات بوت' = chatbot).\n"
     "- A skill counts as known if it, or a closely related tool or technique, "
     "appears anywhere in the context (skills list, projects, certifications, education).\n"
     "- If the user asks about a broad area (e.g. 'data science', 'AI', 'programming'), "
     "list the relevant skills from the context.\n"
     "- If the user asks what I can do or know, summarize my main skills and projects "
     "in a short organized answer.\n\n"

     "Rules for 'do you do/know X?' questions:\n"
     "- If X, or something closely related to X, appears in the context: answer yes "
     "and give the specific details (tools, projects, results).\n"
     "- If X is listed under 'What I do NOT do': say clearly it is not part of my services, "
     "in one friendly sentence, and mention one related thing I do offer.\n"
     "- If X is not mentioned anywhere: say it is not mentioned in my experience, "
     "in one friendly sentence, and mention one related thing I do offer.\n"
     "Do not use a generic refusal for these questions.\n\n"

     "Context:\n{context}"),
    ("human", "Answer in {language} only.\n\nQuestion: {question}"),
])

def format_docs(docs):
    return "\n\n".join(
        f"[{os.path.basename(d.metadata.get('source', ''))}]\n{d.page_content}" for d in docs
    )

In [140]:
def in_scope(question: str) -> bool:
    msg = f"""Decide if the question is about Ahmed Mohamed Galal: his skills, experience,
projects, education, certifications, contact info, or whether he does or does not offer a
certain service or technology. Questions in Arabic or English are both valid.

His scope:
{SCOPE}

ALWAYS answer YES for:
- Questions asking who Ahmed is or to introduce himself or tell about himself.
- Greetings and "what can you help me with" questions.
- "Do you do X?" or "Do you know X?" questions, even if the real answer is no.
General knowledge, jokes, or unrelated tasks are NO.

Examples:
"انت مين؟" -> YES
"عرفني عن نفسك" -> YES
"Tell me about yourself" -> YES
"Who are you?" -> YES
"What can you do?" -> YES
"Hello" -> YES
"مرحبا" -> YES
"ازاي اتواصل معاك؟" -> YES
"ايه خبرتك في Machine Learning؟" -> YES
"بتعمل تطبيقات موبايل؟" -> YES
"Do you know React?" -> YES
"ايه عاصمة فرنسا؟" -> NO
"What is the capital of France?" -> NO
"احكيلي نكتة" -> NO
"Write me a Python script" -> NO

Answer with exactly one word: YES or NO.
Question: {question}
Answer:"""
    out = llm.invoke(msg).content.strip().upper()
    return out.startswith("YES")

In [141]:
import re

OUT_OF_SCOPE_AR = "عذراً، هذا خارج نطاق خدماتي. أقدر أساعدك في الأسئلة عن خبراتي ومشاريعي."
OUT_OF_SCOPE_EN = "Sorry, this is outside my services. I can help with questions about my skills, experience and projects."

def detect_lang(text: str) -> str:
    return "Arabic" if re.search(r"[\u0600-\u06FF]", text) else "English"
def refusal(question: str) -> str:
    return OUT_OF_SCOPE_AR if re.search(r"[\u0600-\u06FF]", question) else OUT_OF_SCOPE_EN

def chat(question: str) -> str:
    if not in_scope(question):
        return refusal(question)

    lang = detect_lang(question)
    docs = retriever.invoke(question)
    msgs = prompt.invoke({
        "context": SCOPE + "\n\n" + format_docs(docs),
        "question": question,
        "language": lang,
    })
    answer = llm.invoke(msgs).content

    if lang == "English" and re.search(r"[\u0600-\u06FF]", answer):
        answer = llm.invoke(msgs).content
        if re.search(r"[\u0600-\u06FF]", answer):
            return "Sorry, something went wrong. Could you rephrase your question?"
    return answer